# evolve

> Evolve the learned folder over many tasks in the way Hermit evolves a harness adapter: a baseline first, pluggable evolvers, a gate on every proposal, versions and an export.

In [ ]:
#| default_exp evolve

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne

`improve_round` learns from one session. This page learns from many. It follows [Hermit](https://github.com/Infini-AI-Lab/hermit), which trains a harness adapter while the model and the harness stay fixed. Here the adapter is the learned folder.

The loop is:

1. Split the tasks into train and held-out.
2. Score the learned folder as it stands on the held-out tasks. This is the baseline. When the folder is not empty, also score an empty one.
3. Run the train tasks. Turn what went wrong into feedback.
4. An evolver proposes new files from the folder and the feedback.
5. The gate checks the proposal and runs it on the held-out tasks. A score below the current one refuses it.
6. Repeat. A person reviews what is kept on a `pending/` branch, as for any lesson.

A runner is a function `runner(task, files) -> turn`. A task is `{prompt, start}`. `files` is the folder as `{path: text}`, or `None` for no folder. `replayer` builds a runner from Ramabana. A test builds one from a fake.

In [ ]:
#| export
import io, json, random, re, subprocess, tarfile, tempfile
from itertools import count
from fastcore.script import call_parse
from fastcore.xtras import Path

from dhrona.improve import (RAMABANA, ROUND_LIMIT, MIN_RUN, MISSION, LEARNED_MAX, REVIEW, _HIDDEN, scrub, check_learned, delivered, route_cost,
                      findings_md, run_turn, replay_clone, round_cost, keep, learned_repo, learned_lock, committed_copy, trainable_sessions, _files, _git)

EVOLVED = 'Evolved learned folder'   #: how the message of a commit made by `evolve_keep` begins

## Scores

`utility` scores one turn from 0 to 1. A turn that gave no answer scores 0. Otherwise the score is 1 over one plus its route findings and failed calls. `score` is the mean over turns. `split_tasks` shuffles with a seed and holds out at least one task, so the same seed gives the same split.

In [ ]:
#| export
def utility(turn):
    "Score of one turn in 0..1: 0 when it gave no answer, else 1 over one plus its findings and failed calls."
    findings, failed, _ = route_cost(turn)
    return delivered(turn) / (1 + findings + failed)

def score(turns):
    "Mean `utility` of `turns`; 0 for none."
    u = [utility(t) for t in turns]
    return sum(u) / len(u) if u else 0.

def split_tasks(tasks, held=.3, seed=0):
    "`(train, held-out)` tasks, shuffled by `seed`; the held-out part is at least one task."
    if len(tasks) < 2: raise ValueError('evolving needs two tasks: one to train on and one held out')
    t = list(tasks); random.Random(seed).shuffle(t)
    n = min(max(1, round(len(t) * held)), len(t) - 1)
    return t[n:], t[:n]

In [ ]:
def turn(*acts, reply='done'): return {'prompt': 'sizes', 'reply': reply, 'usage': {'turns': 1}, 'activity': [dict(tool=t, ok=ok, args=a) for t, ok, a in acts]}
good = turn(('ls', True, {'path': ''}))
bad = turn(('ls', True, {'path': ''}), ('read_file', False, {'path': 'x'}))
assert utility(good) == 1 and 0 < utility(bad) < 1
test_eq(utility(turn(reply='')), 0)
test_eq(score([good, turn(reply='')]), .5); test_eq(score([]), 0)
tr, ho = split_tasks(list(range(10)), .3, 1)
test_eq((len(tr), len(ho)), (7, 3)); test_eq(split_tasks(list(range(10)), .3, 1), (tr, ho)); test_eq(sorted(tr + ho), list(range(10)))
test_eq(len(split_tasks([1, 2])[1]), 1)
test_fail(lambda: split_tasks([1]), contains='two tasks')

## Runners

`write_files` writes `{path: text}` into a folder. `replayer` makes a runner that replays a task in a clone of the repository, with the files as the replay's learned folder. It keeps the total cost under `budget` and raises `BudgetSpent` when less than `MIN_RUN` is left. `chatter` asks the model one question and returns the reply. Both use `run_turn`, so `--model claude/...` sends the work through Claude Code.

In [ ]:
#| export
class BudgetSpent(RuntimeError): "The budget leaves less than `MIN_RUN` for another run."

def write_files(files, dest):
    "Write `{path: text}` under `dest` and return `dest`."
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    for rel, text in files.items(): (dest/rel).parent.mkdir(parents=True, exist_ok=True); (dest/rel).write_text(text)
    return dest

def _left(work, budget):
    "Dollars of `budget` left after the runs in `work`; raises `BudgetSpent` under `MIN_RUN`; None for no budget."
    if budget is None: return None
    if (left := budget - round_cost(work)) < MIN_RUN: raise BudgetSpent(f'${left:.2f} of the ${budget:g} budget is left, under the ${MIN_RUN:g} a run needs')
    return left

def replayer(work, model=None, exe=RAMABANA, approve='edits', timeout=1800, read_outside=True, budget=None):
    "A runner that replays task `{prompt, start}` under learned `files` in a new folder of `work`."
    n = count()
    def run(task, files=None):
        d = Path(work)/f'run-{next(n):03d}'
        return run_turn(task['prompt'], [replay_clone(task['start'], d/'repo')], d/'cfg', model, write_files(files, d/'learned') if files else None,
                        approve, exe, timeout, read_outside, _left(work, budget))
    return run

def chatter(work, model=None, exe=RAMABANA, timeout=900, budget=None):
    "A `chat(prompt) -> reply` that asks the model one question in a new folder of `work`."
    n = count()
    def chat(prompt):
        d = Path(work)/f'chat-{next(n):03d}'; d.mkdir(parents=True)
        t = run_turn(prompt, [d], d/'cfg', model, None, 'ask', exe, timeout, max_cost=_left(work, budget))
        if not delivered(t): raise RuntimeError(f"the model gave no answer: {t.get('error') or t.get('reply') or ''}")
        return t['reply']
    return chat

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = write_files({'AGENTS.md': 'a\n', 'skills/x/SKILL.md': 'b\n'}, Path(d)/'f')
    test_eq(_files(p), {'AGENTS.md': 'a\n', 'skills/x/SKILL.md': 'b\n'})
    test_eq(_left(d, None), None)
    test_fail(lambda: _left(d, .1), contains='budget')
    test_eq(_left(d, 5), 5)

## Evolvers

An evolver is a function `(files, feedback) -> files`. `@evolver` registers a factory under its name in `EVOLVERS`; the factory takes what the evolver needs and returns that function. Two are built in:

- `reflect` shows a model the folder and the feedback, and takes the files it rewrites. A reply of `KEEP` changes nothing. This is the reflective method of GEPA.
- `agent` lets an editor agent edit a copy of the folder, as `improve_round` does.

To add a method, write a factory and decorate it.

In [ ]:
#| export
EVOLVERS = {}   #: name -> factory that returns an evolver

def evolver(f):
    "Register factory `f` in `EVOLVERS` under its name."
    EVOLVERS[f.__name__] = f
    return f

REFLECT = """You improve a coding agent by rewriting its learned folder: `AGENTS.md`, short rules put in every briefing, and `skills/<name>/SKILL.md`, skills opened on demand. A skill starts with frontmatter that holds `name: <name>` and a `description`.

Below are the folder as it stands and feedback from runs under it. The runs were replays with nobody present, so a refused shell command is not a defect. Act only on behaviour that repeats across runs: a failed call, a shell command where a dedicated tool does the job, a repeated call.
- Keep `AGENTS.md` under {mx} characters.
- Write nothing specific to one request or one repository.
- Write no personal detail. `[EMAIL_1]` and similar stand for removed details.
- Change how the agent works, never what it delivers.

# Folder

{folder}

# Feedback

{feedback}

Reply with the single word KEEP if no change is justified. Otherwise reply with each file you change, in full: `FILE: <path>` on its own line, then the whole file in a fenced block."""

_FILE = re.compile(r'^FILE: *`?(\S+?)`? *\n\s*```[^\n]*\n(.*?)\n```[ \t]*$', re.M | re.S)

def parse_files(reply):
    "`{path: text}` for each `FILE:` block in `reply`; empty when there is none, as for KEEP."
    return {p: t + '\n' for p, t in _FILE.findall(reply)}

def render(files):
    "`files` as `FILE:` blocks, the shape `parse_files` reads back."
    return '\n\n'.join(f'FILE: {p}\n```\n{t.rstrip()}\n```' for p, t in files.items()) or '(the folder is empty)'

@evolver
def reflect(chat):
    "Reflective rewrite: `chat(prompt) -> reply` sees the folder and the feedback and returns the files it changes."
    def propose(files, feedback):
        reply = chat(REFLECT.format(mx=LEARNED_MAX, folder=render(files), feedback=feedback))
        if not (new := parse_files(reply)) and 'KEEP' not in reply: raise RuntimeError(f'the reply held no FILE block: {reply[:200]}')
        return files | new
    return propose

@evolver
def agent(work, editor=None, exe=RAMABANA):
    "Agent edit: an editor agent edits a copy of the folder with its tools."
    n = count()
    def propose(files, feedback):
        d = Path(work)/f'edit-{next(n):03d}'
        cand = write_files(files, d/'candidate')
        ev = write_files({'findings.md': feedback, 'task.md': 'Replays of the tasks named in findings.md.'}, d/'evidence')
        t = run_turn(MISSION.format(candidate=cand, evidence=ev, mx=LEARNED_MAX), [cand, ev], d/'cfg', editor, approve='edits', exe=exe)
        if not delivered(t): raise RuntimeError(f"the editor gave no answer: {t.get('error') or t.get('reply') or ''}")
        return _files(cand)
    return propose

In [ ]:
skill = '---\nname: sizes\ndescription: Compare file sizes.\n---\nUse ls.\n'
files = {'AGENTS.md': 'Use `ls`.\n', 'skills/sizes/SKILL.md': skill}
test_eq(parse_files(render(files)), files)
test_eq(parse_files('KEEP'), {})
reply = 'Done.\nFILE: `AGENTS.md`\n```markdown\nNew rule.\n```\n'
test_eq(parse_files(reply), {'AGENTS.md': 'New rule.\n'})
seen = []
r = EVOLVERS['reflect'](lambda p: seen.append(p) or reply)
test_eq(r(files, 'fb'), files | {'AGENTS.md': 'New rule.\n'})
assert 'Use `ls`.' in seen[0] and 'fb' in seen[0] and '2000' in seen[0]
test_eq(EVOLVERS['reflect'](lambda p: 'KEEP')(files, 'fb'), files)
test_eq(parse_files('FILE: AGENTS.md\n\n```markdown\nSpaced.\n```'), {'AGENTS.md': 'Spaced.\n'})   # a blank line before the fence
test_fail(lambda: EVOLVERS['reflect'](lambda p: 'I would add a rule')(files, 'fb'), contains='no FILE block')
assert set(EVOLVERS) >= {'reflect', 'agent'}

## Feedback and the gate

`feedback_md` is what an evolver reads: each train task with its score and findings, then why earlier proposals were refused. It passes through `scrub`, so the evolver sees no personal detail.

`gate` lists reasons to refuse a candidate: a problem `check_learned` finds, or a held-out score that fell below the current one by more than `tol`. A candidate with no reasons passes.

In [ ]:
#| export
def feedback_md(tasks, turns, notes=()):
    "Markdown for an evolver: each task's request, score and findings, then why earlier proposals were refused."
    rows = [f"## Task {i}: {' '.join(t['prompt'].split())[:200]}\n\nscore {utility(u):.2f}\n\n{findings_md([u])}" for i, (t, u) in enumerate(zip(tasks, turns), 1)]
    if notes: rows.append('## Refused earlier\n\n' + '\n'.join(f'- {n}' for n in notes))
    return scrub('\n\n'.join(rows))

def gate(files, before, after, tol=0.):
    "Reasons to refuse candidate `files`: folder problems, or a held-out score `after` under `before` by more than `tol`."
    with tempfile.TemporaryDirectory() as d: bad = check_learned(write_files(files, d))
    return bad + ([f'the held-out score fell from {before:.2f} to {after:.2f}'] if after < before - tol else [])

In [ ]:
fb = feedback_md([{'prompt': 'list  sizes'}, {'prompt': 'mail bob@example.com'}], [good, bad], ['the held-out score fell'])
assert 'Task 1: list sizes' in fb and 'score 1.00' in fb and '## Refused earlier' in fb and 'bob@example.com' not in fb
test_eq(gate(files, .5, .5), [])
test_eq(gate(files, .5, .4, tol=.2), [])
assert 'fell from 0.50 to 0.40' in gate(files, .5, .4)[0]
assert 'neither' in gate({'notes.txt': 'x'}, .5, .5)[0]
assert gate({'AGENTS.md': 'x' * 3000}, 0, 1)

## The loop

`evolve` runs the loop. Each round it runs the train tasks under the current folder, builds feedback, asks the evolver, masks personal values in the proposal, and gates it on the held-out tasks. A refused proposal leaves the folder as it was, and its reasons go into the next feedback. An evolver that returns the folder unchanged ends the loop. A `RuntimeError` in a round, such as `BudgetSpent`, ends the loop too, and the best folder so far stays.

The report holds the held-out score of an empty folder (`none`), of the starting folder (`baseline`) and of the final folder (`final`), the `lineage` of rounds, and the final `files`. `adopted` is true only when `final` beats `baseline`.

In [ ]:
#| export
def evolve(runner, propose, tasks, files=None, rounds=3, held=.3, seed=0, tol=0.):
    "Evolve `files` over `tasks` for up to `rounds`; returns `{adopted, files, none, baseline, final, lineage}`."
    train, hold = split_tasks(tasks, held, seed)
    start = cur = dict(files or {})
    base = score(runner(t, cur or None) for t in hold)
    none = score(runner(t, None) for t in hold) if cur else base
    best, notes, lineage = base, [], [dict(version=0, held=base)]
    for v in range(1, rounds + 1):
        try:
            turns = [runner(t, cur or None) for t in train]
            cand = {p: scrub(t) for p, t in propose(cur, feedback_md(train, turns, notes)).items()}
            if cand == cur: lineage.append(dict(version=v, train=score(turns), note='the evolver changed nothing')); break
            s = score(runner(t, cand) for t in hold)
        except RuntimeError as e: lineage.append(dict(version=v, stopped=str(e))); break
        why = gate(cand, best, s, tol)
        lineage.append(dict(version=v, train=score(turns), held=s, refused=why or None))
        if why: notes += why
        else: cur, best = cand, s
    return dict(adopted=cur != start and best > base, files=cur, none=none, baseline=base, final=best, lineage=lineage)

In [ ]:
def fake(files, **_):
    "A runner whose turns fail unless the folder says to use `ls`."
    return good if 'use ls' in (files or {}).get('AGENTS.md', '') else bad
run = lambda t, files=None: fake(files)
tasks = [{'prompt': f'task {i} about file sizes'} for i in range(6)]
fix = lambda files, fb: files | {'AGENTS.md': 'use ls\n'}
r = evolve(run, fix, tasks, rounds=2)
assert r['adopted'] and r['final'] > r['baseline'] == r['none']
test_eq(r['files'], {'AGENTS.md': 'use ls\n'}); test_eq(len(r['lineage']), 3)
test_eq(r['lineage'][2]['note'], 'the evolver changed nothing')   # round 2: the folder already says it
worse = lambda files, fb: {'AGENTS.md': 'avoid ls\n'}
r = evolve(run, worse, tasks, {'AGENTS.md': 'use ls\n'}, rounds=2)
assert not r['adopted'] and r['final'] == r['baseline'] == 1 and r['none'] < 1
assert 'fell' in r['lineage'][1]['refused'][0]
seen = []
r = evolve(run, lambda f, fb: seen.append(fb) or worse(f, fb), tasks, {'AGENTS.md': 'use ls\n'}, rounds=2)
assert 'Refused earlier' in seen[1] and 'Refused earlier' not in seen[0]
test_eq(evolve(run, lambda f, fb: f, tasks)['lineage'][-1]['note'], 'the evolver changed nothing')
def boom(task, files=None): raise BudgetSpent('out')
n = []
def spent(task, files=None): n.append(1); return fake(files) if len(n) < 8 else boom(task)
r = evolve(spent, fix, tasks, rounds=3)
assert r['lineage'][-1]['stopped'] == 'out' and not r['adopted']
test_fail(lambda: evolve(boom, fix, tasks), contains='out')   # the baseline must run
test_eq(evolve(run, lambda f, fb: {'AGENTS.md': 'use ls. mail a@b.co\n'}, tasks)['files']['AGENTS.md'].count('@'), 0)

## Versions

`evolve_keep` puts an adopted report's files on a `pending/` branch of the learned folder, with the scores in the commit message. Nothing reaches HEAD until a person accepts it. `versions` lists every commit of the folder with the branch it is on. `restore` proposes the folder as it was at one commit, on a new `pending/` branch, so a rollback is reviewed like any lesson.

In [ ]:
#| export
def _current(learned):
    "The files at `learned`'s HEAD, or none when the folder is missing."
    if not Path(learned).exists(): return {}
    with tempfile.TemporaryDirectory() as d: return _files(committed_copy(learned, Path(d)/'c'))

def evolve_keep(learned, report):
    "Commit an adopted `report`'s files on a new `pending/` branch of `learned` and return it; None when nothing was adopted."
    if not report['adopted']: return None
    steps = '\n'.join(f"- round {l['version']}: held-out {l.get('held', 0):.2f}" + (f", refused: {'; '.join(l['refused'])}" if l.get('refused') else '') for l in report['lineage'][1:])
    msg = f"{EVOLVED}: held-out {report['baseline']:.2f} to {report['final']:.2f}\n\nAn empty folder scored {report['none']:.2f}.\n{steps}"
    with tempfile.TemporaryDirectory() as d: return keep(write_files(report['files'], Path(d)/'cand'), learned, msg)

def versions(learned):
    "Every commit of `learned`, newest first: `{rev, branch, at, subject}`; `branch` is the `pending/` branch holding it, else ''."
    rows = _git(learned_repo(learned), 'log', '--all', '--topo-order', '--format=%h%x09%D%x09%aI%x09%s').splitlines()
    return [dict(rev=r, branch=next((b.strip() for b in refs.split(',') if b.strip().startswith('pending/')), ''), at=at, subject=s)
            for r, refs, at, s in (row.split('\t', 3) for row in rows)]

def _snapshot(learned, rev, dest):
    "`learned` as it was at `rev`, extracted into the new folder `dest`."
    dest = Path(dest); dest.mkdir(parents=True)
    if _git(learned, 'ls-tree', '-r', '--name-only', rev):
        tarfile.open(fileobj=io.BytesIO(subprocess.run(['git', '-C', str(learned), 'archive', rev], stdin=subprocess.DEVNULL, capture_output=True, check=True).stdout)).extractall(dest, filter='data')
    return dest

def restore(learned, rev):
    "Propose the folder as it was at `rev` on a new `pending/` branch and return it."
    learned = learned_repo(learned)
    with tempfile.TemporaryDirectory() as d: return keep(_snapshot(learned, rev, Path(d)/'old'), learned, f'Restore the learned folder to {rev}')

In [ ]:
with tempfile.TemporaryDirectory() as d:
    learned = Path(d)/'learned'; learned.mkdir()
    (learned/'AGENTS.md').write_text('old rule\n'); learned_repo(learned)
    test_eq(evolve_keep(learned, dict(adopted=False)), None)
    r = evolve(run, fix, tasks, {'AGENTS.md': 'old rule\n'}, rounds=1)
    b = evolve_keep(learned, r)
    assert b.startswith('pending/')
    v = versions(learned)
    test_eq((len(v), v[0]['branch'], v[0]['subject'].split(':')[0]), (2, b, EVOLVED))
    assert 'An empty folder scored' in _git(learned, 'log', '-1', '--format=%B', b)
    test_eq(_git(learned, 'show', f'{b}:AGENTS.md'), 'use ls')
    old = v[-1]['rev']
    b2 = restore(learned, old)
    test_eq(_git(learned, 'show', f'{b2}:AGENTS.md'), 'old rule')
    test_eq((learned/'AGENTS.md').read_text(), 'old rule\n')   # HEAD stays until a person accepts

## Export

`export_workspace` writes the learned folder's HEAD where other agents read it: `AGENTS.md` for Codex, and for Claude Code a `CLAUDE.md` that imports it plus the skills under `.claude/skills/`. Dated lesson files join `AGENTS.md`.

In [ ]:
#| export
def export_workspace(learned, dest):
    "Write `learned`'s HEAD into `dest` as `AGENTS.md`, `CLAUDE.md` and `.claude/skills/`; returns the paths written."
    files = _current(learned)
    rules = '\n'.join(t for p, t in files.items() if p == 'AGENTS.md' or re.fullmatch(r'dhrona-[\d-]+\.md', p))
    out = ({'AGENTS.md': rules, 'CLAUDE.md': '@AGENTS.md\n'} if rules else {}) | {f'.claude/{p}': t for p, t in files.items() if p.startswith('skills/')}
    write_files(out, dest)
    return sorted(out)

In [ ]:
with tempfile.TemporaryDirectory() as d:
    learned = Path(d)/'l'; write_files(files | {'dhrona-2026-01-02.md': '## lesson\n'}, learned); learned_repo(learned)
    got = export_workspace(learned, Path(d)/'out')
    test_eq(got, ['.claude/skills/sizes/SKILL.md', 'AGENTS.md', 'CLAUDE.md'])
    assert 'lesson' in (Path(d)/'out'/'AGENTS.md').read_text()
    test_eq((Path(d)/'out'/'CLAUDE.md').read_text(), '@AGENTS.md\n')
    test_eq(export_workspace(Path(d)/'empty', Path(d)/'o2'), [])

## Command line

`dhrona-evolve` reads tasks from Ramabana's history (or a `--tasks` JSON file of `{prompt, start}`), runs `evolve` under a budget, writes `report.json` and puts what it adopts on a `pending/` branch. Pass `--model claude/<model>` to run through Claude Code. `dhrona-versions` lists the commits, or proposes a restore. `dhrona-export-learned` exports the folder.

In [ ]:
#| export
def _factory(method, work, editor, exe, budget):
    "The evolver `method` with what it needs, built over `work`."
    kw = dict(reflect=lambda: dict(chat=chatter(work, editor, exe, budget=budget)), agent=lambda: dict(work=work, editor=editor, exe=exe))
    if method not in kw: raise ValueError(f"method is one of {sorted(kw)}, not {method!r}")
    return EVOLVERS[method](**kw[method]())

@call_parse
def evolve_cli(
    work:str,                       # a new folder for the runs and `report.json`
    learned:str,                    # the learned folder to start from; an adopted result waits on its `pending/` branch
    n:int=8,                        # tasks taken from Ramabana's history
    rounds:int=3,                   # rounds of propose, gate and keep
    method:str='reflect',           # `reflect` or `agent`, or a name added with `@evolver`
    model:str=None,                 # the model the replays run with; `claude/<model>` runs through Claude Code
    editor:str=None,                # the model that proposes; None uses `model`
    held:float=.3,                  # fraction of tasks held out
    seed:int=0,                     # the split's seed
    tasks:str=None,                 # a JSON file of `{prompt, start}` instead of the history
    cfg:str='~/.config/ramabana',   # the Ramabana config folder whose history is read
    max_calls:int=40,               # take only sessions whose first turn made at most this many calls
    approve:str='edits',            # `ask` refuses every write in a replay
    exe:str=RAMABANA,               # the Ramabana command a run uses
    budget:float=20.,               # dollars: no run starts once the work has cost this much
    timeout:int=1800,               # seconds before a replay is abandoned
):
    "Evolve the learned folder over past tasks, print one line per round and what it cost."
    work = Path(work)
    if work.exists(): raise FileExistsError(f'{work} exists; an evolve writes into a new folder')
    work.mkdir(parents=True)
    ts = json.loads(Path(tasks).read_text()) if tasks else [{k: v for k, v in s.items() if k in ('prompt', 'start')} for s in trainable_sessions(cfg, max_calls)[:n]]
    editor = editor or model
    with learned_lock(learned):
        r = evolve(replayer(work, model, exe, approve, timeout, budget=budget), _factory(method, work, editor, exe, budget), ts, _current(learned), rounds, held, seed)
        r['pending'] = evolve_keep(learned, r)
    r['cost'] = round_cost(work)
    (work/'report.json').write_text(json.dumps(r, indent=1, default=str))
    for l in r['lineage']: print(l)
    print(f"held-out: empty {r['none']:.2f}, start {r['baseline']:.2f}, final {r['final']:.2f}; spent ${r['cost']:.2f}")
    print(f"pending {r['pending']}\n{REVIEW}" if r['pending'] else 'nothing kept')

@call_parse
def versions_cli(
    learned:str,        # the learned folder
    restore_to:str=None, # a revision from the list: propose the folder as it was then
):
    "List the commits of the learned folder, or propose a restore to one of them."
    if restore_to: return print(f'pending {restore(learned, restore_to)}\n{REVIEW}')
    for v in versions(learned): print(f"{v['rev']}  {v['at'][:10]}  {v['branch'] or 'main':<28} {v['subject'][:70]}")

@call_parse
def export_cli(
    learned:str,  # the learned folder
    dest:str,     # the workspace to write `AGENTS.md`, `CLAUDE.md` and `.claude/skills/` into
):
    "Export the learned folder's HEAD for Codex and Claude Code."
    for p in export_workspace(learned, dest): print(p)

In [ ]:
test_eq(sorted(EVOLVERS), ['agent', 'reflect'])
test_fail(lambda: _factory('nope', '.', None, 'ramabana', None), contains='method is one of')
test_eq(_current('/nonexistent/learned'), {})
with tempfile.TemporaryDirectory() as d:
    test_fail(lambda: evolve_cli(d, d), contains='exists')